In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, auc
from sklearn.model_selection import train_test_split

from rapidgbm import RapidGBMTuner

import glob
import mplhep as hep
hep.style.use([hep.style.ATLAS])
import pickle

import uproot

In [2]:
def unpair_df(data):
    """
    Unpairs columns in the DataFrame that start with 'l1' or 'l2' by renaming them to 'l'
    and concatenating the resulting DataFrames vertically.

    Parameters:
    data (pd.DataFrame): The original DataFrame containing columns to be unpaired.

    Returns:
    pd.DataFrame: A new DataFrame with columns starting with 'l1' and 'l2' renamed to 'l',
                  and the DataFrame length doubled by concatenating the modified DataFrames.
    """
    l1_columns = [col for col in data.columns if 'l1' in col]
    l2_columns = [col for col in data.columns if 'l2' in col]
    print('1')
    other_columns = [col for col in data.columns if not ('l1' in col or 'l2' in col)]
    print('2')
    # Create a copy of the DataFrame with 'l1' and 'l2' columns renamed to 'l'
    data_l1 = data[other_columns + l1_columns].copy()
    data_l2 = data[other_columns + l2_columns].copy()
    print('3')
    data_l1.rename(columns={col: col.replace('l1', 'm_lx', 1) for col in l1_columns}, inplace=True)
    data_l2.rename(columns={col: col.replace('l2', 'm_lx', 1) for col in l2_columns}, inplace=True)
    print('4')
    # Concatenate the DataFrames vertically
    new_data = pd.concat([data_l1, data_l2], ignore_index=True)
    return new_data

def repair_df(data):
    """
    Pairs columns in the DataFrame that contain 'm_lx' by renaming them back to 'l1' and 'l2',
    and merges rows based on shared columns, resulting in the original DataFrame format.

    Parameters:
    data (pd.DataFrame): The DataFrame output from `unpair_df` function with doubled rows.

    Returns:
    pd.DataFrame: A DataFrame where columns containing 'm_lx' are renamed back to 'l1' and 'l2',
                  and rows are merged back to their original format with shared columns appearing only once.
    """
    # Split the data into two halves
    midpoint = len(data) // 2
    data_l1 = data.iloc[:midpoint].copy()
    data_l2 = data.iloc[midpoint:].copy()

    # Identify shared columns (not containing 'm_lx')
    shared_columns = [col for col in data.columns if 'm_lx' not in col]

    # Rename columns by replacing 'm_lx' with 'l1' in data_l1 and 'l2' in data_l2
    data_l1.rename(columns={col: col.replace('m_lx', 'l1') for col in data_l1.columns if 'm_lx' in col}, inplace=True)
    data_l2.rename(columns={col: col.replace('m_lx', 'l2') for col in data_l2.columns if 'm_lx' in col}, inplace=True)

    # Concatenate the two halves on columns, only including shared columns once
    paired_data = pd.concat(
        [data_l1.reset_index(drop=True)[shared_columns + [col for col in data_l1.columns if col not in shared_columns]],
         data_l2.reset_index(drop=True)[[col for col in data_l2.columns if col not in shared_columns]]],
        axis=1
    )

    return paired_data

def filter_columns(df, key_phrases):
    """
    Filters out columns that contain any of the key phrases in their names.

    Parameters:
    df (pd.DataFrame): The dataframe to filter.
    key_phrases (list): A list of key phrases to filter out.

    Returns:
    list: A list of column names that do not contain any of the key phrases.
    """
    filtered_columns = [col for col in df.columns if not any(phrase in col for phrase in key_phrases)]
    return filtered_columns


In [3]:
# # use glob to import all the files
# filePath = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zee_PhPy_601189_8Nov_hist/'

# # Use glob to search for all .root files recursively in the directory and its subdirectories
# root_files = glob.glob(f'{filePath}/**/*.root', recursive=True)

# size_zee = 0
# # Print the list of .root files
# for file in root_files:
#     zee_file = uproot.open(file)
#     zee_data = zee_file['HZG_Tree']
#     size_zee += len(zee_data["EventInfo.cutflow"].array())
#     print(len(zee_data["EventInfo.cutflow"].array()))
# print(f'number of saved zee-events: {size_zee}')

# filePath = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.ttbar_PhPy_601589_11Nov_hist/'

# root_files = glob.glob(f'{filePath}/**/*.root', recursive=True)

# size_ttbar = 0
# for file in root_files:
#     ttbar_file = uproot.open(file)
#     ttbar_data = ttbar_file['HZG_Tree']
#     size_ttbar += len(ttbar_data["EventInfo.cutflow"].array())
#     print(len(ttbar_data["EventInfo.cutflow"].array()))
# print(f'number of saved ttbar-events: {size_ttbar}')

In [4]:
filePath_zee = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zee_PhPy_601189_8Nov_hist/'
filePath_ttbar = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.ttbar_PhPy_601589_11Nov_hist/'
filePath_zmm = '/groups/hep/kinch/data/preprocessed_data/files_from_grid/user.amoranch.Zmm_PhPy_601190_11Nov_hist/'

In [5]:
root_files_zee = glob.glob(f'{filePath_zee}/**/*.root', recursive=True)
root_files_ttbar = glob.glob(f'{filePath_ttbar}/**/*.root', recursive=True)
root_files_zmm = glob.glob(f'{filePath_zmm}/**/*.root', recursive=True)


for file in root_files_zee:
    #check if .parquet file already exists
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    zee_file = uproot.open(file)
    zee_data = zee_file['HZG_Tree']
    data_zee = pd.DataFrame.from_dict({key: zee_data[key].array() for key in zee_data.keys()})
    data_zee.to_parquet(f'{file[:-5]}.parquet')

for file in root_files_ttbar:
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    ttbar_file = uproot.open(file)
    ttbar_data = ttbar_file['HZG_Tree']
    data_ttbar = pd.DataFrame.from_dict({key: ttbar_data[key].array() for key in ttbar_data.keys()})
    data_ttbar.to_parquet(f'{file[:-5]}.parquet')

for file in root_files_zmm:
    if os.path.exists(f'{file[:-5]}.parquet'):
        continue
    zmm_file = uproot.open(file)
    zmm_data = zmm_file['HZG_Tree']
    data_zmm = pd.DataFrame.from_dict({key: zmm_data[key].array() for key in zmm_data.keys()})
    data_zmm.to_parquet(f'{file[:-5]}.parquet')


In [6]:
parqeuet_files_zee = glob.glob(f'{filePath_zee}/**/*.parquet', recursive=True)
parqeuet_files_ttbar = glob.glob(f'{filePath_ttbar}/**/*.parquet', recursive=True)

data_zee = pd.concat([unpair_df(pd.read_parquet(file)) for file in parqeuet_files_zee])
data_ttbar = pd.concat([unpair_df(pd.read_parquet(file)) for file in parqeuet_files_ttbar])

1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4
1
2
3
4


In [7]:
# from_z_bool = (data_zee['l1_truthOrigin'] == 13) & (data_zee['l2_truthOrigin'] == 13) & (data_zee['l1_truthPdgId'] * data_zee['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_zee['truth_from_Z'] = from_z_bool

# from_z_bool = (data_ttbar['l1_truthOrigin'] == 13) & (data_ttbar['l2_truthOrigin'] == 13) & (data_ttbar['l1_truthPdgId'] * data_ttbar['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_ttbar['truth_from_Z'] = from_z_bool

In [8]:
# print(f'number of events in Zee: {data_zee.shape[0]}, ratio of isolated electrons: {(len(data_zee[data_zee['l1_truthType']==2]) + len(data_zee[data_zee['l2_truthType']==2]))/(data_zee.shape[0]*2)}')
# print(f'number of events in ttbar: {data_ttbar.shape[0]}, ratio of isolated electrons: {(len(data_ttbar[data_ttbar['l1_truthType']==2]) + len(data_ttbar[data_ttbar['l2_truthType']==2]))/(data_ttbar.shape[0]*2)}')

# if data_zee.shape[0] > data_ttbar.shape[0]:
#     data_zee = data_zee.sample(n=data_ttbar.shape[0])
# else:
#     data_ttbar = data_ttbar.sample(n=data_zee.shape[0])

# Data_BalZ = pd.concat([data_zee, data_ttbar])
# Data_BalZ.to_parquet('Data_BalZ.parquet')

In [ ]:
# data_zee = pd.concat([pd.read_parquet(file) for file in parqeuet_files_zee])
# data_ttbar = pd.concat([pd.read_parquet(file) for file in parqeuet_files_ttbar])

# from_z_bool = (data_zee['l1_truthOrigin'] == 13) & (data_zee['l2_truthOrigin'] == 13) & (data_zee['l1_truthPdgId'] * data_zee['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_zee['truth_from_Z'] = from_z_bool

# from_z_bool = (data_ttbar['l1_truthOrigin'] == 13) & (data_ttbar['l2_truthOrigin'] == 13) & (data_ttbar['l1_truthPdgId'] * data_ttbar['l2_truthPdgId'] == -121)
# print(from_z_bool.sum()/len(from_z_bool))

# data_ttbar['truth_from_Z'] = from_z_bool

In [ ]:
# Data = pd.concat([data_zee, data_ttbar])
# print(f'ratio of isolated electrons: {(len(Data[Data['l1_truthType']==2]) + len(Data[Data['l2_truthType']==2]))/(Data.shape[0]*2)}')

In [ ]:
Data_unpaired = pd.concat([data_zee, data_ttbar])

In [13]:
print(Data_unpaired.columns)

Index(['Btag60_N_j', 'Btag70_N_j', 'Btag77_N_j', 'Btag85_N_j', 'Btag_N_j',
       'Central_N_j', 'Central_pT_jj', 'EventInfo.PVx', 'EventInfo.PVy',
       'EventInfo.PVz',
       ...
       'm_lx_istruth', 'm_lx_istruth2', 'm_lx_istruth_2', 'm_lx_passid_Loose',
       'm_lx_passid_Medium', 'm_lx_passiso_FCLoose',
       'm_lx_passiso_Loose_VarRad', 'm_lx_passiso_PflowLoose_FixedRad',
       'm_lx_passiso_TightTrackOnly_VarRad', 'l1_neflowisom_lx0'],
      dtype='object', length=367)


In [14]:
# Data_unpaired_zee = unpair_df(data_zee)
# data_zee = None
print('got here')
# Data_unpaired_ttbar = unpair_df(data_ttbar)
# data_ttbar = None
print('got here')
data_zee = None
data_ttbar = None
print('got here')
Data_unpaired_iso = Data_unpaired[Data_unpaired['m_lx_truthType'] == 2]
Data_unpaired_noniso = Data_unpaired[Data_unpaired['m_lx_truthType'] != 2]

print(f'number of isolated electrons: {Data_unpaired_iso.shape[0]}, number of non-isolated electrons: {Data_unpaired_noniso.shape[0]}')

if Data_unpaired_iso.shape[0] > Data_unpaired_noniso.shape[0]:
    Data_unpaired_iso = Data_unpaired_iso.sample(n=Data_unpaired_noniso.shape[0])
elif Data_unpaired_iso.shape[0] < Data_unpaired_noniso.shape[0]:
    Data_unpaired_noniso = Data_unpaired_noniso.sample(n=Data_unpaired_iso.shape[0])

got here
got here
got here
number of isolated electrons: 56731423, number of non-isolated electrons: 6789269


In [15]:
Data_BalIso = pd.concat([Data_unpaired_iso, Data_unpaired_noniso])

Data_BalIso.to_parquet('Data_BalIso.parquet')

: 